In [29]:
import os
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import rdkit 
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import IPythonConsole

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, accuracy_score
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, Subset, TensorDataset

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [31]:
_ = torch.manual_seed(42)

In [32]:
with Chem.SDMolSupplier('bindingDB_mgDB_processed_dataset.sdf') as suppl:
    molecules = [mol for mol in suppl]
#
data_dict = {}
for mol in molecules:
    smiles = mol.GetProp('Molecule (Canonical)')
    label = mol.GetProp('Class')
    data_dict[smiles] = label
#
df_small_act = pd.DataFrame.from_dict(data_dict, orient='index', columns=['label'])
df_small_act.reset_index(inplace=True)
df_small_act = df_small_act.rename(columns={'index': 'smiles'})
df_small_act.loc[df_small_act['label'] == 'active', 'activity_label'] = int(1)
df_small_act.loc[df_small_act['label'] == 'inactive', 'activity_label'] = int(0)
df_small_act

,smiles,label,activity_label
0,O=C1CNC(=O)N1,active,1.0
1,CC(=O)NC(N)=O,active,1.0
2,O=c1cc[nH]c(=O)[nH]1,active,1.0
3,O=C1CCCC(=O)N1,active,1.0
4,CN1CC(=O)NC1=O,active,1.0
...,...,...,...
864,CSc1ccc(C2(NC(=O)Nc3ccc4c(c3)CN(C3CCC(=O)NC3=O...,inactive,0.0
865,O=C1CCC(N2Cc3c(NC(=O)COCCOCCOCc4ccccc4)cccc3C2...,inactive,0.0
866,O=C1CCC(N2Cc3cc(NC(=O)COCCOCCOCc4ccccc4)ccc3C2...,inactive,0.0
867,O=C1CCC(N2C(=O)c3ccc(OCCOCCOCCOCc4ccccc4)cc3C2...,inactive,0.0


In [33]:
df_small_act['label'].value_counts()

label
active      697
inactive    172
Name: count, dtype: int64

In [34]:
with Chem.SDMolSupplier('bindingDB_mgDB_processed_dataset.sdf') as suppl:
    molecules = [mol for mol in suppl]
#
data_dict = {}
for mol in molecules:
    smiles = mol.GetProp('Molecule (Canonical)')
    label = mol.GetProp('ModeOfAction')
    data_dict[smiles] = label
#
df_small_cov = pd.DataFrame.from_dict(data_dict, orient='index', columns=['label'])
df_small_cov.reset_index(inplace=True)
df_small_cov = df_small_cov.rename(columns={'index': 'smiles'})
df_small_cov.loc[df_small_cov['label'] == 'covalent', 'covalency_label'] = int(1)
df_small_cov.loc[df_small_cov['label'] == 'non-covalent', 'covalency_label'] = int(0)
df_small_cov

,smiles,label,covalency_label
0,O=C1CNC(=O)N1,non-covalent,0.0
1,CC(=O)NC(N)=O,non-covalent,0.0
2,O=c1cc[nH]c(=O)[nH]1,non-covalent,0.0
3,O=C1CCCC(=O)N1,non-covalent,0.0
4,CN1CC(=O)NC1=O,non-covalent,0.0
...,...,...,...
864,CSc1ccc(C2(NC(=O)Nc3ccc4c(c3)CN(C3CCC(=O)NC3=O...,non-covalent,0.0
865,O=C1CCC(N2Cc3c(NC(=O)COCCOCCOCc4ccccc4)cccc3C2...,non-covalent,0.0
866,O=C1CCC(N2Cc3cc(NC(=O)COCCOCCOCc4ccccc4)ccc3C2...,non-covalent,0.0
867,O=C1CCC(N2C(=O)c3ccc(OCCOCCOCCOCc4ccccc4)cc3C2...,non-covalent,0.0


In [35]:
df_small_cov['label'].value_counts()

label
non-covalent    853
covalent         16
Name: count, dtype: int64

In [36]:
with Chem.SDMolSupplier('enamine_processed_dataset.sdf') as suppl:
    molecules = [mol for mol in suppl]
#
data_dict = {}
for mol in molecules:
    smiles = mol.GetProp('Molecule')
    label = mol.GetProp('Mode of Action')
    data_dict[smiles] = label
#
df_large_cov = pd.DataFrame.from_dict(data_dict, orient='index', columns=['label'])
df_large_cov.reset_index(inplace=True)
df_large_cov = df_large_cov.rename(columns={'index': 'smiles'})
df_large_cov.loc[df_large_cov['label'] == 'Covalent', 'cov_label'] = int(1)
df_large_cov.loc[df_large_cov['label'] == 'Noncovalent', 'cov_label'] = int(0)
df_large_cov

,smiles,label,cov_label
0,O=C1CCC(N2Cc3ccc(C(=O)O)cc3C2=O)C(=O)N1,Noncovalent,0.0
1,O=C1CCC(N2Cc3cc(C(=O)O)ccc3C2=O)C(=O)N1,Noncovalent,0.0
2,CNC(=O)c1ccc2c(c1)CN(C1CCC(=O)NC1=O)C2=O,Noncovalent,0.0
3,CCNC(=O)c1ccc2c(c1)CN(C1CCC(=O)NC1=O)C2=O,Noncovalent,0.0
4,O=C1CCC(N2Cc3cc(C(=O)NC4CC4)ccc3C2=O)C(=O)N1,Noncovalent,0.0
...,...,...,...
3484,Cc1ccc(C(=O)NC23CC(C2)C(=O)NC3=O)c(Br)c1,Noncovalent,0.0
3485,N#CC1(NC(=O)Cc2cc(=O)[nH]c(=O)[nH]2)CCCCC1,Covalent,1.0
3486,C=CC(=O)N1CCCC(n2ccc(=O)[nH]c2=O)C1,Covalent,1.0
3487,Cn1c(N)c(N2C(=O)C=CC2=O)c(=O)[nH]c1=O,Covalent,1.0


In [37]:
df_large_cov['label'].value_counts()

label
Noncovalent    3354
Covalent        135
Name: count, dtype: int64

In [38]:
with Chem.SDMolSupplier('enamine_processed_dataset.sdf') as suppl:
    molecules = [mol for mol in suppl]
#
data_dict = {}
for mol in molecules:
    smiles = mol.GetProp('Molecule')
    label = float(mol.GetProp('MW (desalted)'))
    data_dict[smiles] = label
#
df_large_frag = pd.DataFrame.from_dict(data_dict, orient='index', columns=['label'])
df_large_frag.reset_index(inplace=True)
df_large_frag = df_large_frag.rename(columns={'index': 'smiles'})
df_large_frag.loc[df_large_frag['label'] < 350, 'frag_label'] = int(1)
df_large_frag.loc[df_large_frag['label'] >= 350, 'frag_label'] = int(0)
df_large_frag.sort_values(by='label', inplace=True)
df_large_frag

,smiles,label,frag_label
3479,O=C1NC(=O)[C@]2(c3ccccc3)C[C@H]12,187.195,1.0
1286,O=C1CC(c2ccccc2)CC(=O)N1,189.211,1.0
3383,O=C1CC(c2cccnc2)CC(=O)N1,190.199,1.0
2614,O=C1CCC(c2ccccn2)C(=O)N1,190.199,1.0
3386,O=C1CC(c2cccs2)CC(=O)N1,195.240,1.0
...,...,...,...
3027,Nc1sc2c(c1C(=O)Nc1ccc(NC3CCC(=O)NC3=O)cc1)CCNC2,399.469,0.0
2293,CN1CCc2nc(C(=O)Nc3ccc(NC4CCC(=O)NC4=O)cc3)sc2C1,399.469,0.0
3253,CC(C)CC(C(=O)Nc1cccc(C2CCC(=O)NC2=O)c1)N1CCCCC1=O,399.484,0.0
3040,O=C1CCC(Nc2ccc(NC(=O)C3CCCC4(CCCCC4)O3)cc2)C(=...,399.484,0.0


In [39]:
df_large_frag['frag_label'].value_counts()

frag_label
1.0    1749
0.0    1740
Name: count, dtype: int64

In [40]:
from rdkit.Chem import Descriptors

with Chem.SDMolSupplier('bindingDB_mgDB_processed_dataset.sdf') as suppl:
    molecules = [mol for mol in suppl]
#
data_dict = {}
for mol in molecules:
    smiles = mol.GetProp('Molecule (Canonical)')
    label = float(Descriptors.MolWt(mol))
    data_dict[smiles] = label
#
df_small_frag = pd.DataFrame.from_dict(data_dict, orient='index', columns=['label'])
df_small_frag.reset_index(inplace=True)
df_small_frag = df_small_frag.rename(columns={'index': 'smiles'})
df_small_frag.loc[df_small_frag['label'] < 350, 'frag_label'] = int(1)
df_small_frag.loc[df_small_frag['label'] >= 350, 'frag_label'] = int(0)
df_small_frag.sort_values(by='label', inplace=True)
df_small_frag

,smiles,label,frag_label
0,O=C1CNC(=O)N1,100.077,1.0
1,CC(=O)NC(N)=O,102.093,1.0
2,O=c1cc[nH]c(=O)[nH]1,112.088,1.0
3,O=C1CCCC(=O)N1,113.116,1.0
4,CN1CC(=O)NC1=O,114.104,1.0
...,...,...,...
692,Cn1nc(C2CCC(=O)NC2=O)c2ccc(NC(=O)NC3(CNC(=O)OC...,498.584,0.0
693,O=C1CCC(N2Cc3cc(-c4cc(CN5CCCC5)c5[nH]nc(C6CCC6...,498.587,0.0
694,O=C1CCC(N2Cc3cc(CNC(=O)C4=Cc5cccc(C(F)(F)F)c5O...,499.445,0.0
695,CN(c1ccccc1)c1nccc(NCC(=O)Nc2cccc3c2CN(C2CCC(=...,499.531,0.0


In [41]:
df_small_frag['frag_label'].value_counts()

frag_label
0.0    536
1.0    333
Name: count, dtype: int64

In [42]:
class SmilesTokenizer(object):
    def __init__(self):
        self.regex_pattern = (
            r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\."
            r"|=|#|-|\+|\\|\/|:|~|@|\?|>>?|\*|\$|\%[0-9]{2}|[0-9])"
        )
        self.regex = re.compile(self.regex_pattern)

    def tokenize(self, smiles):
        tokens = [token for token in self.regex.findall(smiles)]
        return tokens

In [43]:
def build_vocab(smiles_list, tokenizer, max_vocab_size):
    tokenized_smiles = [tokenizer.tokenize(s) for s in smiles_list]
    token_counter = Counter(c for s in tokenized_smiles for c in s)
    tokens = [token for token, _ in token_counter.most_common(max_vocab_size)]
    vocab = {token: idx for idx, token in enumerate(tokens)}
    return vocab


def smiles_to_ohe(smiles, tokenizer, vocab):
    unknown_token_id = len(vocab) - 1
    token_ids = [vocab.get(token, unknown_token_id) for token in tokenizer.tokenize(smiles)]
    ohe = torch.eye(len(vocab))[token_ids]
    return ohe

In [44]:
tokenizer = SmilesTokenizer()

In [45]:
smiles = df_large_frag['smiles'].iloc[0]
print("SMILES string:\n\t", smiles)
print("Tokens:\n\t", ", ".join(tokenizer.tokenize(smiles)))
vocab = build_vocab([smiles], tokenizer, 30)
print("Vocab:\n\t", vocab)
print("Shape of OHE matrix:\n", np.array(smiles_to_ohe(smiles, tokenizer, vocab)).T.shape)

SMILES string:
	 O=C1NC(=O)[C@]2(c3ccccc3)C[C@H]12
Tokens:
	 O, =, C, 1, N, C, (, =, O, ), [C@], 2, (, c, 3, c, c, c, c, c, 3, ), C, [C@H], 1, 2
Vocab:
	 {'c': 0, 'C': 1, 'O': 2, '=': 3, '1': 4, '(': 5, ')': 6, '2': 7, '3': 8, 'N': 9, '[C@]': 10, '[C@H]': 11}
Shape of OHE matrix:
 (12, 26)


/tmp/ipykernel_84812/633349569.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print("Shape of OHE matrix:\n", np.array(smiles_to_ohe(smiles, tokenizer, vocab)).T.shape)


In [46]:
smiles = df_large_frag['smiles'].iloc[-1]
print("SMILES string:\n\t", smiles)
print("Tokens:\n\t", ", ".join(tokenizer.tokenize(smiles)))
vocab = build_vocab([smiles], tokenizer, 30)
print("Vocab:\n\t", vocab)
#print("OHE:\n", np.array(smiles_to_ohe(smiles, tokenizer, vocab)).T)
print("Shape of OHE matrix:\n", np.array(smiles_to_ohe(smiles, tokenizer, vocab)).T.shape)

SMILES string:
	 CC1CCN(S(=O)(=O)c2ccc(C(=O)N[C@@H]3CCC(=O)NC3=O)s2)CC1
Tokens:
	 C, C, 1, C, C, N, (, S, (, =, O, ), (, =, O, ), c, 2, c, c, c, (, C, (, =, O, ), N, [C@@H], 3, C, C, C, (, =, O, ), N, C, 3, =, O, ), s, 2, ), C, C, 1
Vocab:
	 {'C': 0, '(': 1, ')': 2, '=': 3, 'O': 4, 'c': 5, 'N': 6, '1': 7, '2': 8, '3': 9, 'S': 10, '[C@@H]': 11, 's': 12}
Shape of OHE matrix:
 (13, 49)


/tmp/ipykernel_84812/2703194191.py:7: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print("Shape of OHE matrix:\n", np.array(smiles_to_ohe(smiles, tokenizer, vocab)).T.shape)


In [47]:
def dataprep_for_GRU(input_df, smiles_col, label_col):
    n_samples = len(input_df)
    df = input_df.sample(frac=1, random_state=42).reset_index(drop=True)
    smiles = df[smiles_col].tolist()
    y = df[label_col].to_numpy().astype(np.float32)
    #
    test_ratio = 0.1
    val_ratio = 0.1
    train_ratio = 1.0 - test_ratio - val_ratio
    #
    indices = np.arange(n_samples)
    ids_train_val, ids_test, y_train_val, y_test = train_test_split(
        indices,
        y,
        test_size=test_ratio,
        random_state=42,
        stratify=y
    )
    #
    relative_val_ratio = val_ratio / (train_ratio + val_ratio) # 0.1 / 0.9 ≈ 0.111
    #
    ids_train, ids_val, y_train, y_val = train_test_split(
        ids_train_val,
        y_train_val,
        test_size=relative_val_ratio,
        random_state=42,
        stratify=y_train_val
    )
    #
    print("## 🧪 Stratification Check")
    print(f"Total samples: {n_samples}")
    print(f"Train samples: {len(ids_train)} ({len(ids_train)/n_samples:.1%}) | Positive Ratio: {y[ids_train].mean():.4f}")
    print(f"Validation samples: {len(ids_val)} ({len(ids_val)/n_samples:.1%}) | Positive Ratio: {y[ids_val].mean():.4f}")
    print(f"Test samples: {len(ids_test)} ({len(ids_test)/n_samples:.1%}) | Positive Ratio: {y[ids_test].mean():.4f}")
    print(f"Overall Positive Ratio: {y.mean():.4f}")
    #
    max_vocab_size = 50
    smiles_train = [smiles[i] for i in ids_train]
    vocab = build_vocab(smiles_train, tokenizer, max_vocab_size)
    print("Vocab:\n\t", vocab)
    vocab_size = len(vocab)
    print("Vocab size:\n\t", vocab_size)
    #
    X = pad_sequence(
        sequences=[smiles_to_ohe(smi, tokenizer, vocab) for smi in smiles],
        batch_first=True,
        padding_value=0,
    )
    #
    data = TensorDataset(X, torch.tensor(y, dtype=torch.float32))
    #
    train_loader = DataLoader(
        Subset(data, ids_train),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(42),
    )
    #
    val_loader = DataLoader(
        Subset(data, ids_val),
        batch_size=64,
        shuffle=True,
        generator=torch.Generator().manual_seed(42)
    )
    #
    test_loader = DataLoader(
        Subset(data, ids_test),
        batch_size=1,
        shuffle=False,
        generator=torch.Generator().manual_seed(42),
    )
    return train_loader, val_loader, test_loader, vocab_size

In [48]:
train_loader, val_loader, test_loader, vocab_size = dataprep_for_GRU(
    df_large_frag,
    smiles_col='smiles',
    label_col='frag_label'
)

## 🧪 Stratification Check
Total samples: 3489
Train samples: 2791 (80.0%) | Positive Ratio: 0.5013
Validation samples: 349 (10.0%) | Positive Ratio: 0.5014
Test samples: 349 (10.0%) | Positive Ratio: 0.5014
Overall Positive Ratio: 0.5013
Vocab:
	 {'C': 0, 'c': 1, '(': 2, ')': 3, 'O': 4, '=': 5, '1': 6, 'N': 7, '2': 8, '3': 9, 'n': 10, '4': 11, 'F': 12, '-': 13, '[nH]': 14, 's': 15, 'Cl': 16, 'o': 17, '5': 18, 'S': 19, 'Br': 20, '[C@@H]': 21, '#': 22, '[C@H]': 23, '[N+]': 24, '[O-]': 25, '/': 26, '[C@]': 27, '[2H]': 28, '6': 29, '\\': 30}
Vocab size:
	 31


In [49]:
class GRUClassificationModel(nn.Module):
    """GRU network for binary classification"""

    def __init__(self, input_size, hidden_size=32, num_layers=1):
        """
        GRU network
        Parameters
        ----------
        input_size : int
            The number of expected features in the input vector (i.e., vocab_size).
        hidden_size : int
            The number of features in the hidden state.
        num_layers : int
            The number of recurrent layers (default is 1).
        """
        super(GRUClassificationModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # 1. GRU Layer (same as regression)
        # Processes the sequence of one-hot encoded tokens.
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        
        # 2. Final Linear Layer: Maps the hidden state to a single logit.
        # The output size is 1 for binary classification.
        self.fc = nn.Linear(hidden_size, 1)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(p=0.2)
        
        # NOTE: Sigmoid is typically applied in the training loop with BCEWithLogitsLoss 
        # or applied here and then use nn.BCELoss, but nn.BCEWithLogitsLoss is preferred 
        # for numerical stability. We'll leave it out of the forward pass for now.

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size/vocab_size)
        
        # Initialize hidden state h0
        # If running on GPU, you'll need to send the tensor to the correct device (e.g., .to(x.device))
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # GRU forward pass
        # out shape: (batch_size, sequence_length, hidden_size)
        # hn shape: (num_layers, batch_size, hidden_size)
        out, hn = self.gru(x, h0)
        
        # 1. Take the hidden state of the LAST token in the sequence
        # out[:, -1] shape: (batch_size, hidden_size)
        out = out[:, -1]
        
        # 2. Apply Dropout
        out = self.dropout(out)
        
        # 3. Final Linear Layer: maps hidden state to logit (unbounded value)
        # out shape: (batch_size, 1)
        out = self.fc(out)
        
        return out

In [50]:
class ModelTrainerBCEloss(object):
    """A class that provides training and validation infrastructure for the model and keeps track of training and validation metrics."""

    def __init__(self, model, lr, clip_gradients=False):
        """
        Initialization.

        Parameters
        ----------
        model : nn.Module
            a model
        lr : float
            learning rate for one training step

        """
        self.model = model
        self.lr = lr
        self.criterion = torch.nn.BCEWithLogitsLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), self.lr)
        self.clip_gradients = clip_gradients
        self.model.to(device)

        self.train_loss = []
        self.batch_loss = []
        self.val_loss = []

    def _train_epoch(self, loader):
        self.model.train()
        epoch_loss = 0
        batch_losses = []
        for i, (X_batch, y_batch) in enumerate(loader):
            # Move inputs and targets to the correct device and ensure target dtype matches loss
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).float()
            self.optimizer.zero_grad()
            y_pred = self.model(X_batch)
            loss = self.criterion(y_pred, y_batch.unsqueeze(1))
            loss.backward()

            if self.clip_gradients:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1, norm_type=2)

            self.optimizer.step()
            epoch_loss += loss.item()
            batch_losses.append(loss.item())

        return epoch_loss / len(loader), batch_losses

    def _eval_epoch(self, loader):
        self.model.eval()
        val_loss = 0
        probabilities = [] # Changed from 'predictions'
        targets = []
        with torch.no_grad():
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device).float()
                
                y_pred_logits = self.model(X_batch) # This is the logit (unbounded value)
                loss = self.criterion(y_pred_logits, y_batch.unsqueeze(1))
                val_loss += loss.item()
                
                # Convert logits to probabilities
                y_pred_probs = torch.sigmoid(y_pred_logits)
                
                # Append probabilities and targets
                probabilities.append(y_pred_probs.detach().cpu().numpy()) # Store probabilities
                targets.append(y_batch.unsqueeze(1).detach().cpu().numpy())

        probabilities = np.concatenate(probabilities).flatten() # Flatten 
        targets = np.concatenate(targets).flatten()
        
        # Calculate Matthews correlation coefficient
        mcc = matthews_corrcoef(targets, (probabilities >= 0.5).astype(int))

        # Convert probabilities to binary predictions for accuracy
        binary_preds = (probabilities >= 0.5).astype(int)
        accuracy = accuracy_score(targets, binary_preds)
        
        # Return loss, probabilities (for full analysis), targets, and key metrics
        return val_loss / len(loader), probabilities, targets, mcc, accuracy

    def train(self, train_loader, val_loader, n_epochs, print_every=10):
        # Initialize lists to track metrics
        self.val_mcc = []
        self.val_accuracy = []
        
        for e in range(n_epochs):
            # 1. Training Step
            train_loss, train_loss_batches = self._train_epoch(train_loader)
            
            # 2. Validation Step (Updated call to get new metrics)
            val_loss, _, _, val_mcc, val_accuracy = self._eval_epoch(val_loader)
            
            # 3. Store results
            self.batch_loss += train_loss_batches
            self.train_loss.append(train_loss)
            self.val_loss.append(val_loss)
            self.val_mcc.append(val_mcc)
            self.val_accuracy.append(val_accuracy)
            
            if e % print_every == 0:
                print(
                    f"Epoch {e+1:03} | train_loss: {train_loss:.5f} | val_loss: {val_loss:.5f} "
                    f"| val_MCC: {val_mcc:.4f} | val_ACC: {val_accuracy:.4f}" # Print new metrics
                )

    def validate(self, val_loader):
        """
        Validate the model

        Parameters
        ----------
        val_loader :
            a dataloader with validation data

        Returns
        -------
        Tuple[float, list, list, float, float]
            Loss, y_probabilities, y_target, AUC, Accuracy
        """
        # Call the updated _eval_epoch
        loss, y_probs, y_targ, mcc, accuracy = self._eval_epoch(val_loader)
        
        # Return all results, including the new metrics
        return loss, y_probs, y_targ, mcc, accuracy

In [51]:
model_gru = ModelTrainerBCEloss(
    model=GRUClassificationModel(vocab_size, hidden_size=32),
    lr=1e-4,
    clip_gradients=True
)
model_gru.train(train_loader, val_loader, 101)
test_loss, test_probs, test_targets, test_mcc, test_accuracy = model_gru.validate(test_loader)
print("## 🧪 Final Test Metrics")
print("-" * 30)
print(f"Test Set Count: {len(test_targets)}")
print(f"Test Loss (BCEWithLogits): {test_loss:.5f}")
print(f"Test MCC: {test_mcc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("-" * 30)
#
print("\n## 📊 Classification Report")
binary_predictions = (test_probs >= 0.5).astype(int)
print(classification_report(test_targets, binary_predictions, target_names=['Class 0 (Inactive)', 'Class 1 (Active)']))

Epoch 001 | train_loss: 0.69527 | val_loss: 0.69483 | val_MCC: 0.0000 | val_ACC: 0.4986
Epoch 011 | train_loss: 0.64523 | val_loss: 0.60903 | val_MCC: 0.7442 | val_ACC: 0.8711
Epoch 021 | train_loss: 0.36745 | val_loss: 0.35934 | val_MCC: 0.7457 | val_ACC: 0.8711
Epoch 031 | train_loss: 0.31613 | val_loss: 0.31458 | val_MCC: 0.7476 | val_ACC: 0.8711
Epoch 041 | train_loss: 0.27376 | val_loss: 0.25897 | val_MCC: 0.8060 | val_ACC: 0.9026
Epoch 051 | train_loss: 0.22576 | val_loss: 0.23625 | val_MCC: 0.8193 | val_ACC: 0.9083
Epoch 061 | train_loss: 0.18679 | val_loss: 0.21481 | val_MCC: 0.8308 | val_ACC: 0.9140
Epoch 071 | train_loss: 0.16888 | val_loss: 0.16911 | val_MCC: 0.8741 | val_ACC: 0.9370
Epoch 081 | train_loss: 0.15141 | val_loss: 0.15060 | val_MCC: 0.8859 | val_ACC: 0.9427
Epoch 091 | train_loss: 0.14359 | val_loss: 0.14137 | val_MCC: 0.8915 | val_ACC: 0.9456
Epoch 101 | train_loss: 0.15185 | val_loss: 0.17438 | val_MCC: 0.8567 | val_ACC: 0.9255
## 🧪 Final Test Metrics
--------

In [52]:
def dataprep_for_GRU_nosplit(input_df, smiles_col, label_col):
    df = input_df.sample(frac=1, random_state=42).reset_index(drop=True)
    smiles = df[smiles_col].tolist()
    y = df[label_col].to_numpy().astype(np.float32)
    #
    max_vocab_size = 50
    vocab = build_vocab(smiles, tokenizer, max_vocab_size)
    print("Vocab:\n\t", vocab)
    vocab_size = len(vocab)
    print("Vocab size:\n\t", vocab_size)
    #
    X = pad_sequence(
        sequences=[smiles_to_ohe(smi, tokenizer, vocab) for smi in smiles],
        batch_first=True,
        padding_value=0,
    )
    #
    data = TensorDataset(X, torch.tensor(y, dtype=torch.float32))
    #
    test_loader = DataLoader(
        data,
        batch_size=1,
        shuffle=False,
        generator=torch.Generator().manual_seed(42),
    )
    return test_loader

small_test_loader = dataprep_for_GRU_nosplit(df_small_frag, 'smiles', 'frag_label')
                                             

Vocab:
	 {'c': 0, 'C': 1, '(': 2, ')': 3, 'O': 4, '=': 5, 'N': 6, '1': 7, '2': 8, '3': 9, '4': 10, 'n': 11, '5': 12, 'F': 13, '-': 14, 'Cl': 15, 'S': 16, '/': 17, '[C@H]': 18, 's': 19, '6': 20, '[nH]': 21, '[C@@H]': 22, '#': 23, '[N+]': 24, '[O-]': 25, 'o': 26, 'Br': 27, '7': 28, '\\': 29, '[C@@]': 30, '[NH3+]': 31, '[2H]': 32, '[C@]': 33, '[NH2+]': 34, 'I': 35, 'P': 36}
Vocab size:
	 37


In [53]:
test_loss, test_probs, test_targets, test_mcc, test_accuracy = model_gru.validate(small_test_loader)
print("## 🧪 Final Test Metrics")
print("-" * 30)
print(f"Test Set Count: {len(test_targets)}")
print(f"Test Loss (BCEWithLogits): {test_loss:.5f}")
print(f"Test MCC: {test_mcc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("-" * 30)
#
print("\n## 📊 Classification Report")
binary_predictions = (test_probs >= 0.5).astype(int)
print(classification_report(test_targets, binary_predictions, target_names=['Class 0 (Inactive)', 'Class 1 (Active)']))

RuntimeError: input.size(-1) must be equal to input_size. Expected 31, got 37

In [22]:
train_loader, val_loader, test_loader, vocab_size = dataprep_for_GRU(
    df_large_cov,
    smiles_col='smiles',
    label_col='cov_label'
)

## 🧪 Stratification Check
Total samples: 3489
Train samples: 2791 (80.0%) | Positive Ratio: 0.0387
Validation samples: 349 (10.0%) | Positive Ratio: 0.0372
Test samples: 349 (10.0%) | Positive Ratio: 0.0401
Overall Positive Ratio: 0.0387
Vocab:
	 {'C': 0, 'c': 1, '(': 2, ')': 3, 'O': 4, '=': 5, '1': 6, 'N': 7, '2': 8, '3': 9, 'n': 10, '4': 11, 'F': 12, '-': 13, '[nH]': 14, 's': 15, 'o': 16, 'Cl': 17, 'S': 18, '5': 19, 'Br': 20, '[C@@H]': 21, '[C@H]': 22, '#': 23, '[N+]': 24, '[O-]': 25, '/': 26, '[C@]': 27, '[2H]': 28, '6': 29, '\\': 30, '[C@@]': 31}
Vocab size:
	 32


In [23]:
model_gru = ModelTrainerBCEloss(
    model=GRUClassificationModel(vocab_size, hidden_size=32),
    lr=1e-4,
    clip_gradients=True
)
model_gru.train(train_loader, val_loader, 101)
test_loss, test_probs, test_targets, test_mcc, test_accuracy = model_gru.validate(test_loader)
print("## 🧪 Final Test Metrics")
print("-" * 30)
print(f"Test Set Count: {len(test_targets)}")
print(f"Test Loss (BCEWithLogits): {test_loss:.5f}")
print(f"Test MCC: {test_mcc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("-" * 30)
#
print("\n## 📊 Classification Report")
binary_predictions = (test_probs >= 0.5).astype(int)
print(classification_report(test_targets, binary_predictions, target_names=['Class 0 (Noncovalent)', 'Class 1 (Covalent)']))

Epoch 001 | train_loss: 0.59340 | val_loss: 0.58157 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 011 | train_loss: 0.17124 | val_loss: 0.16123 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 021 | train_loss: 0.16677 | val_loss: 0.15821 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 031 | train_loss: 0.16499 | val_loss: 0.16827 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 041 | train_loss: 0.16563 | val_loss: 0.15798 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 051 | train_loss: 0.16436 | val_loss: 0.16797 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 061 | train_loss: 0.16209 | val_loss: 0.16772 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 071 | train_loss: 0.16603 | val_loss: 0.15717 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 081 | train_loss: 0.16448 | val_loss: 0.15637 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 091 | train_loss: 0.16024 | val_loss: 0.15472 | val_MCC: 0.0000 | val_ACC: 0.9628
Epoch 101 | train_loss: 0.15398 | val_loss: 0.14728 | val_MCC: 0.0000 | val_ACC: 0.9628
## 🧪 Final Test Metrics
--------

/home/haolan/anaconda3/envs/deepchem/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/haolan/anaconda3/envs/deepchem/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/haolan/anaconda3/envs/deepchem/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me